# Paper 4 Validation: Single-Coder Two-Pass Methodology

This notebook implements the single-coder validation approach from the SCEN-EC paper adapted for Paper 4's 10-strategy codebook.

**Structure:**
- **Section A**: Draw stratified validation sample (run once)
- **CODING BREAK**: You code the Excel file offline in two passes (strict, permissive)
- **Section B**: Compute Cohen's kappa metrics for reporting

**Total sentences to code**: 220 (20 per category × 10 categories + 20 unlabeled)
**Coding load**: 220 sentences × 2 passes = 440 codings

## Section A: Draw Validation Sample

**Run this once.** It produces `validation_sample.xlsx` with an Instructions sheet and a Sample sheet ready for coding.

In [ ]:
# Cell A1: Setup and paths

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import pandas as pd
import numpy as np
from pathlib import Path

# ---- ADAPT THESE PATHS ----
# Path to the sentence-level labeled data produced by Paper4_OptionC_Analysis.ipynb
SENTENCES_CSV = '/content/drive/MyDrive/Colab Notebooks/IFResearch/Strategy/paper4_optionC/sentences_labeled.csv'

# Output directory for validation materials
OUT_DIR = Path('/content/drive/MyDrive/Colab Notebooks/IFResearch/Strategy/paper4_optionC/validation')
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_XLSX = OUT_DIR / 'validation_sample.xlsx'
OUT_KAPPA = OUT_DIR / 'kappa_results.xlsx'

# ---- FIXED PARAMETERS ----
N_PER_CATEGORY = 20
N_UNLABELED = 20
SEED = 42

STRATEGY_CATEGORIES = [
    'pivot', 'experimentation', 'customer_discovery', 'resource_substitution',
    'sequencing', 'scaling', 'optimization', 'capability_building',
    'partnering', 'legitimation'
]

# Sentence length filter (chars) — excludes filler and garbled sentences
MIN_CHARS = 40
MAX_CHARS = 400

print('Setup complete.')

In [ ]:
# Cell A2: Load and inspect input data

df = pd.read_csv(SENTENCES_CSV)
print(f'Total sentences: {len(df):,}')
print(f'Columns: {list(df.columns)}')
print()
print('First 3 rows:')
display(df.head(3))

# ---- HANDLE strategies LIST COLUMN ----
# Your sentences CSV has a 'strategies' column containing Python lists
# e.g. [] or ['strategy_partnering'] or ['strategy_pivot', 'strategy_scaling']
# We extract the first strategy from each list as the primary label.

import ast

def extract_primary(val):
    """Extract first strategy from the strategies list column."""
    if pd.isna(val):
        return ''
    s = str(val).strip()
    if s in ('[]', '', 'nan'):
        return ''
    try:
        lst = ast.literal_eval(s)
        if isinstance(lst, list) and len(lst) > 0:
            # Strip the 'strategy_' prefix to match codebook category names
            label = str(lst[0]).strip()
            label = label.replace('strategy_', '')
            return label
        return ''
    except Exception:
        # Fallback: plain string
        label = s.replace('strategy_', '').strip("[]'\" ")
        return label

df['strategy_label'] = df['strategies'].apply(extract_primary)

print(f'\nExtracted strategy_label from strategies column.')
print(f'Label distribution (top 15):')
print(df['strategy_label'].value_counts().head(15))
print(f'\nUnlabeled sentences: {(df["strategy_label"] == "").sum():,}')

# Verify required columns now exist
required = {'sentence_id', 'episode_id', 'text', 'strategy_label'}
missing = required - set(df.columns)
assert not missing, f'Still missing: {missing}'
print('\nAll required columns present. Ready for sampling.')


In [ ]:
# Cell A3: Draw stratified sample

# Apply length filter
df['n_chars'] = df['text'].str.len()
df_valid = df[(df['n_chars'] >= MIN_CHARS) & (df['n_chars'] <= MAX_CHARS)].copy()
print(f'After length filter ({MIN_CHARS}–{MAX_CHARS} chars): {len(df_valid):,} sentences')

rng = np.random.default_rng(SEED)
sample_frames = []

# Sample 20 per strategy category
for cat in STRATEGY_CATEGORIES:
    pool = df_valid[df_valid['strategy_label'] == cat]
    n_available = len(pool)
    n_draw = min(N_PER_CATEGORY, n_available)
    if n_draw < N_PER_CATEGORY:
        print(f'  WARNING: {cat} has only {n_available} sentences, drawing all')
    if n_draw > 0:
        drawn = pool.sample(n=n_draw, random_state=SEED)
        drawn = drawn.assign(regex_label=cat)
        sample_frames.append(drawn)
        print(f'  {cat}: sampled {n_draw} of {n_available}')

# Sample 20 unlabeled
unlabeled_pool = df_valid[df_valid['strategy_label'] == '']
print(f'\nUnlabeled pool: {len(unlabeled_pool):,}')
unlabeled_drawn = unlabeled_pool.sample(n=N_UNLABELED, random_state=SEED)
unlabeled_drawn = unlabeled_drawn.assign(regex_label='(none)')
sample_frames.append(unlabeled_drawn)
print(f'  unlabeled: sampled {N_UNLABELED}')

# Combine and shuffle so coder cannot anticipate the regex label
sample = pd.concat(sample_frames, ignore_index=True)
sample = sample.sample(frac=1, random_state=SEED).reset_index(drop=True)
sample.insert(0, 'row_id', range(1, len(sample) + 1))

print(f'\nFinal sample size: {len(sample)}')
print(f'Regex-label distribution in sample:')
print(sample['regex_label'].value_counts())

In [ ]:
# Cell A4: Write validation_sample.xlsx with instructions + sample sheets

# Prepare the coding columns (empty)
sample_out = sample[['row_id', 'sentence_id', 'episode_id', 'text', 'regex_label']].copy()
sample_out['pass_A_label'] = ''  # coder fills
sample_out['pass_B_label'] = ''  # coder fills
sample_out['notes'] = ''          # optional coder notes

# HIDE the regex_label from the coder to reduce anchoring bias
# We save it separately in a hidden reference sheet
reference = sample_out[['row_id', 'regex_label']].copy()
sample_visible = sample_out.drop(columns=['regex_label'])

INSTRUCTIONS = pd.DataFrame({
    'Field': [
        'PURPOSE',
        '',
        'STRATEGY CATEGORIES (10)',
        '',
        'PASS A: STRICT PRIOR',
        'Rule',
        'Label a sentence only if',
        'Ambiguous sentence →',
        'Partial or hedged discourse →',
        'Mixed / multiple candidate strategies →',
        '',
        'PASS B: PERMISSIVE PRIOR',
        'Rule',
        'Label a sentence if',
        'Ambiguous sentence →',
        'Partial or hedged discourse →',
        'Mixed / multiple candidate strategies →',
        '',
        'WORKFLOW',
        'Step 1',
        'Step 2',
        'Step 3',
        'Step 4',
        'Step 5',
        '',
        'LABEL VALUES',
        'For strategy present',
        'For no strategy',
        'Case matters?',
        '',
        'IMPORTANT',
        'Do not consult the regex classifier during coding',
        'Do not consult Pass A while doing Pass B',
    ],
    'Description': [
        'Validate the 10-strategy codebook via two coding passes with deliberately different priors. Cohen\'s kappa between the two passes measures codebook discriminability; kappa against the regex classifier measures pipeline agreement.',
        '',
        'pivot | experimentation | customer_discovery | resource_substitution | sequencing | scaling | optimization | capability_building | partnering | legitimation',
        '',
        '',
        'Prefer no label over a stretched label.',
        'the strategic move is explicit, unambiguous, and directly stated.',
        'leave blank (no label)',
        'leave blank',
        'leave blank (do not pick one arbitrarily)',
        '',
        '',
        'Prefer a label over blank when discourse is plausibly strategic.',
        'any part of the discourse points toward the strategy.',
        'assign the most plausible label',
        'assign the label the sentence gestures toward',
        'assign the dominant one',
        '',
        '',
        'Complete Pass A for all 220 rows before starting Pass B.',
        'Take a break of at least 2 hours between passes to avoid anchoring.',
        'Complete Pass B for all 220 rows.',
        'Do not revisit Pass A after starting Pass B.',
        'Save the file as validation_sample_coded.xlsx before running Section B.',
        '',
        '',
        'Type the exact category name (e.g., partnering).',
        'Leave the cell blank.',
        'No, but spelling must match the category list.',
        '',
        '',
        'The regex_label column is hidden intentionally.',
        'This ensures the two passes are actually independent.',
    ]
})

with pd.ExcelWriter(OUT_XLSX, engine='openpyxl') as writer:
    INSTRUCTIONS.to_excel(writer, sheet_name='Instructions', index=False)
    sample_visible.to_excel(writer, sheet_name='Sample', index=False)
    reference.to_excel(writer, sheet_name='_reference_regex', index=False)

# Hide the reference sheet
import openpyxl
wb = openpyxl.load_workbook(OUT_XLSX)
wb['_reference_regex'].sheet_state = 'hidden'

# Widen the text column in Sample sheet
sample_ws = wb['Sample']
sample_ws.column_dimensions['A'].width = 8   # row_id
sample_ws.column_dimensions['B'].width = 15  # sentence_id
sample_ws.column_dimensions['C'].width = 15  # episode_id
sample_ws.column_dimensions['D'].width = 90  # text
sample_ws.column_dimensions['E'].width = 22  # pass_A_label
sample_ws.column_dimensions['F'].width = 22  # pass_B_label
sample_ws.column_dimensions['G'].width = 30  # notes

# Enable text wrapping on the text column
from openpyxl.styles import Alignment
for row in sample_ws.iter_rows(min_row=2, max_row=sample_ws.max_row, min_col=4, max_col=4):
    for cell in row:
        cell.alignment = Alignment(wrap_text=True, vertical='top')

# Freeze header row
sample_ws.freeze_panes = 'A2'

# Widen Instructions sheet
instr_ws = wb['Instructions']
instr_ws.column_dimensions['A'].width = 45
instr_ws.column_dimensions['B'].width = 100
for row in instr_ws.iter_rows(min_row=2, max_row=instr_ws.max_row, min_col=2, max_col=2):
    for cell in row:
        cell.alignment = Alignment(wrap_text=True, vertical='top')

wb.save(OUT_XLSX)
print(f'Saved: {OUT_XLSX}')
print(f'Size: {OUT_XLSX.stat().st_size / 1024:.1f} KB')

## ▶▶▶ CODING BREAK ◀◀◀

1. Download `validation_sample.xlsx` from Drive
2. Read the Instructions sheet
3. Do **Pass A (strict)** on all 220 rows in the `pass_A_label` column
4. Wait at least 2 hours
5. Do **Pass B (permissive)** on all 220 rows in the `pass_B_label` column
6. Save as `validation_sample_coded.xlsx` back to the validation folder
7. Come back and run Section B below

## Section B: Compute Cohen's Kappa

After you have coded both passes, run this section to produce the four kappa metrics for the paper.

In [ ]:
# Cell B1: Load coded file and merge with regex reference

import pandas as pd
from pathlib import Path
from sklearn.metrics import cohen_kappa_score

OUT_DIR = Path('/content/drive/MyDrive/Colab Notebooks/IFResearch/Strategy/paper4_optionC/validation')
CODED_XLSX = OUT_DIR / 'validation_sample_coded.xlsx'
OUT_KAPPA = OUT_DIR / 'kappa_results.xlsx'

STRATEGY_CATEGORIES = [
    'pivot', 'experimentation', 'customer_discovery', 'resource_substitution',
    'sequencing', 'scaling', 'optimization', 'capability_building',
    'partnering', 'legitimation'
]

# Load the coded sample and the hidden reference
coded = pd.read_excel(CODED_XLSX, sheet_name='Sample')
reference = pd.read_excel(CODED_XLSX, sheet_name='_reference_regex')

# Merge on row_id
merged = coded.merge(reference, on='row_id', how='left')

# Clean and normalize labels
def clean_label(x):
    if pd.isna(x) or str(x).strip() in ('', '(none)', 'none', 'NONE', 'None'):
        return 'none'
    return str(x).strip().lower().replace(' ', '_')

merged['A'] = merged['pass_A_label'].apply(clean_label)
merged['B'] = merged['pass_B_label'].apply(clean_label)
merged['R'] = merged['regex_label'].apply(clean_label)

# Check for typos: any label that's not in the expected set
valid_labels = set(STRATEGY_CATEGORIES) | {'none'}
for col in ['A', 'B']:
    bad = merged[~merged[col].isin(valid_labels)]
    if len(bad):
        print(f'WARNING: Pass {col} has {len(bad)} unrecognized labels:')
        print(bad[["row_id", col]].to_string())
        print('Fix these in the Excel file before proceeding.')

n_coded_A = (merged['A'] != 'none').sum()
n_coded_B = (merged['B'] != 'none').sum()
n_coded_R = (merged['R'] != 'none').sum()
print(f'\nSample size: {len(merged)}')
print(f'Pass A: {n_coded_A} labeled, {len(merged) - n_coded_A} left blank')
print(f'Pass B: {n_coded_B} labeled, {len(merged) - n_coded_B} left blank')
print(f'Regex:  {n_coded_R} labeled, {len(merged) - n_coded_R} unlabeled')

In [ ]:
# Cell B2: Compute overall Cohen's kappa

def compute_kappa(y1, y2, name1, name2):
    k = cohen_kappa_score(y1, y2)
    n = len(y1)
    return {'Comparison': f'{name1} vs {name2}', 'Kappa': round(k, 3), 'n': n}

results = []
results.append(compute_kappa(merged['A'], merged['B'], 'Pass A (strict)', 'Pass B (permissive)'))
results.append(compute_kappa(merged['A'], merged['R'], 'Pass A (strict)', 'Regex pipeline'))
results.append(compute_kappa(merged['B'], merged['R'], 'Pass B (permissive)', 'Regex pipeline'))

def interpret(k):
    if k < 0.20: return 'Poor'
    if k < 0.40: return 'Fair'
    if k < 0.60: return 'Moderate'
    if k < 0.80: return 'Substantial'
    return 'Near-perfect'

for r in results:
    r['Interpretation'] = interpret(r['Kappa'])

overall = pd.DataFrame(results)
print('Overall kappa results:')
display(overall)

In [ ]:
# Cell B3: Per-category kappa
# For each strategy category, compute kappa treating the problem as binary
# (this category vs. everything else)

per_category_rows = []
for cat in STRATEGY_CATEGORIES:
    A_bin = (merged['A'] == cat).astype(int)
    B_bin = (merged['B'] == cat).astype(int)
    R_bin = (merged['R'] == cat).astype(int)
    
    # Only compute if there's some variation (not all zeros or all ones)
    def safe_kappa(x, y):
        if x.sum() == 0 and y.sum() == 0:
            return float('nan')  # no positives in either -> undefined
        return round(cohen_kappa_score(x, y), 3)
    
    row = {
        'Category': cat,
        'n_A': int(A_bin.sum()),
        'n_B': int(B_bin.sum()),
        'n_regex_in_sample': int(R_bin.sum()),
        'kappa_A_vs_B': safe_kappa(A_bin, B_bin),
        'kappa_A_vs_regex': safe_kappa(A_bin, R_bin),
        'kappa_B_vs_regex': safe_kappa(B_bin, R_bin),
    }
    per_category_rows.append(row)

per_category = pd.DataFrame(per_category_rows)
print('Per-category kappa:')
display(per_category)

In [ ]:
# Cell B4: Save all results to a single Excel file for the paper

with pd.ExcelWriter(OUT_KAPPA, engine='openpyxl') as writer:
    overall.to_excel(writer, sheet_name='Overall_Kappa', index=False)
    per_category.to_excel(writer, sheet_name='Per_Category_Kappa', index=False)
    merged[['row_id', 'sentence_id', 'text', 'A', 'B', 'R']].to_excel(
        writer, sheet_name='Full_Coded_Data', index=False
    )

print(f'Saved: {OUT_KAPPA}')
print()
print('=== NUMBERS TO INSERT INTO PAPER 4 SECTION 3.4 ===')
print()
print('OVERALL KAPPA:')
for _, r in overall.iterrows():
    print(f"  {r['Comparison']}: kappa = {r['Kappa']:.3f} (n={r['n']}, {r['Interpretation'].lower()})")
print()
print('PER-CATEGORY (kappa A vs B, top strong / bottom weak):')
sorted_cat = per_category.sort_values('kappa_A_vs_B', ascending=False)
for _, r in sorted_cat.iterrows():
    k = r['kappa_A_vs_B']
    k_str = f"{k:.3f}" if pd.notna(k) else "n/a"
    print(f"  {r['Category']:22s}: kappa_AB = {k_str}")